Dataset structure should be:

<pre>
data
    yes
        ...
    no
        ...

excel
    40753679
        excel_file.xlsx (doesn't matter what the file is called)
    47333462
        excel_file.xlsx (doesn't matter what the file is called)
    47333473
        excel_file.xlsx (doesn't matter what the file is called)

</pre>

In [ ]:
!pip install openpyxl

In [ ]:
import os, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedShuffleSplit

import numpy as np
from PIL import Image

# Used to process the excel data
import pandas as pd
import shutil

In [ ]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        img, label = self.subset[i]  # returns PIL image if base_ds.transform is None
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /2

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /4

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /8
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # global average pool
            nn.Flatten(),
            nn.Dropout(0, num_classes),
            nn.Linear(64, 1),              # binary logit
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x  # raw logits

In [ ]:
def excel_to_data(data_dir):

    # Get all the names in the excel file
    scenes = os.listdir(data_dir)
    names = [[], []] # names[0] all the "yes", names[1] all the "no"
    for s in scenes:
        dir = data_dir + '/' + s
        if len(os.listdir(dir)) > 0:
            excel_file = os.listdir(dir)[0]
            df = pd.read_excel(dir + '/' + excel_file)
            for i in range(len(df["frame_index"])):
                name = s + '_'
                name += df["query"][i] + '_'
                name += str(df["frame_index"][i]) + '_'
                name += "rendered.png"
                if df["object_present"][i] == True:
                    names[0].append(name)
                else:
                    names[1].append(name)
    print(f"Found {len(names[0])} TRUE")
    print(f"Found {len(names[1])} FALSE")
    
    # Copy the files from E_grade_pics to data
    copied = [0, 0]
    queries = os.listdir("E_grade_pics")
    for q in queries:
        pic_names = os.listdir("E_grade_pics/" + q)
        for correct in [0, 1]:
            for name in names[correct]:
                if name in pic_names:
                    copied[correct] += 1
                    source_path = "E_grade_pics/" + q + '/' + name
                    if correct == 0:
                        destination_path = "data/yes/" + name
                    else:
                        destination_path = "data/no/" + name
                    shutil.copyfile(source_path, destination_path)
                elif name[-12:] == "rendered.png":
                    print(f"File {name} not found in excel data")
    
    print(f"Copied {copied[0]} files into yes")
    print(f"Copied {copied[1]} files into no")
    return None

In [ ]:
def qet_queries(data_dir):
    files = []
    for folder in ['yes', 'no']:
        dir = data_dir + '/' + folder
        files += os.listdir(dir)
    
    queries = []
    for f in files:
        q = f.split("_")[1]
        if q not in queries:
            queries.append(q)
    return queries

In [ ]:
def run_epoch(loader, train: bool):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # shape [B,1], values 0/1

        if train:
            optimizer.zero_grad()

        logits = model(imgs)
        loss = criterion(logits, labels)

        if train:
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            total += labels.numel()

        total_loss += loss.item() * labels.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = total_correct / max(total, 1)
    return avg_loss, acc


In [ ]:
def predict_yes_no(image_path: str, model_path: str = "yesno_cnn.pt") -> str:
    """Return exactly 'yes' or 'no' for a single image."""
    ckpt = torch.load(model_path, map_location="cpu")
    image_size = ckpt.get("image_size", 256)
    class_to_idx = ckpt["class_to_idx"]
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    # Rebuild the same TinyCNN
    m = TinyCNN()
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])

    img = Image.open(image_path).convert("RGB")
    x = tfm(img).unsqueeze(0)  # [1,3,H,W]

    with torch.no_grad():
        logit = m(x)
        prob_yes = torch.sigmoid(logit)[0, 0].item()

    # In ImageFolder, 'no' should be class 0 and 'yes' class 1 if folders are named that way.
    # We threshold the probability of class 1 ('yes') at 0.5.
    return "yes" if prob_yes >= 0.5 else "no"

In [ ]:
DATA_DIR = "data"
IMAGE_SIZE = 256            # set to 600 if you want full-res (slower!)
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-3
SEED = 1337

#excel_to_data("excel") # UNCOMMENT this to load the excel data into "data"

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device (CUDA / Apple MPS / CPU)
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Using device:", device)

In [ ]:
base_ds = datasets.ImageFolder(
    DATA_DIR,
    transform=None,  # defer transforms
)

queries = qet_queries(DATA_DIR)
print("List of queries:", queries)

class_names = base_ds.classes  # should be ["no", "yes"] if folders are named like that
print("Classes:", class_names)

# Train/val split (80/20) that's stable with our seed
num_samples = len(base_ds)
indices = list(range(num_samples))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

labels = base_ds.targets  # class index for each sample
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=2),
    transforms.ToTensor(),                # [0,1]
])

val_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
train_base = Subset(base_ds, train_idx)
val_base = Subset(base_ds, val_idx)
train_ds = TransformSubset(train_base, train_tfms)
val_ds   = TransformSubset(val_base,   val_tfms)

# DataLoaders
num_workers = 4 if os.name != "nt" else 0  # Windows -> 0 workers is safest
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=(device.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(device.type == "cuda"))


In [ ]:
#model = TinyCNN(len(queries)).to(device) # Not implemented yet
model = TinyCNN(2).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": base_ds.class_to_idx,
            "image_size": IMAGE_SIZE,
            "arch": "TinyCNN",
        }, "yesno_cnn.pt")
        print("Saved checkpoint: yesno_cnn.pt")

print("Best val acc:", best_val_acc)

In [ ]:
print(predict_yes_no("data/no/40753679_sofa_5_rendered.png"))
print(predict_yes_no("data/yes/40753679_floor_79_rendered.png"))